# partial  —  App G    sparse (feature-level) damage + K-vs-V split

Frozen Qwen2.5-1.5B, split-context SQuAD. Anchors are re-measured
in-run (floor = reader's own masked cache, reference = its unmasked cache).

Results are written to `results/`, figures to `figures/`.


# "But what if most of the features were good?" — sparse vs dense cache damage

The cosine experiment perturbed **every** coordinate of the cache a little and the reader died at
cos 0.995. The natural objection is that this is a pathological stimulus: real cache-production methods
get most features approximately right and only a few badly wrong, and attention ought to route around a
few bad coordinates.

Here we keep a fraction `1-p` of the cache **bit-exact** and destroy only the fraction `p`.

- **P1** — sweep `p` over a random subset of KV features. Graceful in `(1-p)`, or a cliff at tiny `p`?
- **P2** — matched-cosine control: sparse-severe vs dense-mild at the *same* achieved cosine.
- **P3** — structural subsets: exactly one KV **head**, exactly one **layer**.
- **P4** — **K-half vs V-half**: the softmax theory predicts K damage is far worse than V damage.

Anchors, reader, probe and decode path are identical to `similar.ipynb` so every number here is
directly comparable to that experiment's Arm A / Arm B.

In [ ]:
import os
# Write next to this notebook's parent (the repo root), not the CWD, so
# results land in the same results/ that make_figs.py and verify_results.py read.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
import os
CACHE = os.environ.get("HF_HOME") or os.path.expanduser("~/.cache/huggingface")
os.environ["HF_HOME"]=CACHE; os.environ["HF_DATASETS_CACHE"]=CACHE
os.environ["TOKENIZERS_PARALLELISM"]="false"
import json, numpy as np, re
import torch, torch.nn as nn, torch.nn.functional as F
import datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.cache_utils import DynamicCache
import matplotlib.pyplot as plt
try:
    get_ipython().run_line_magic("matplotlib","inline")
except NameError:
    pass  # running as a plain script, not under IPython
assert torch.cuda.is_available(), (
    "This experiment needs a CUDA GPU. The reader runs in fp32 (deliberately: the\n"
    "cosine and attention-logit measurements are sensitive to reduced precision),\n"
    "so expect ~8 GB and more for the larger notebooks.")
DEVICE="cuda"; SEED=0; torch.manual_seed(SEED); np.random.seed(SEED)
print("Device:",torch.cuda.get_device_name(0),flush=True)
BASE="Qwen/Qwen2.5-1.5B"
CFG=dict(n_eval=200, max_ctx=256, gen_len=8,
         # fraction of FEATURES damaged
         fracs=[0.0,0.01,0.05,0.10,0.25,0.50,1.0],
         # dense-perturbation cosine targets for the matched control (Arm A construction)
         cos_targets=[0.9999,0.999,0.995,0.99,0.95,0.90],
         seeds=[0,1,2])
if os.environ.get("KVL_FAST","0")=="1":
    CFG.update(n_eval=24, seeds=[0]); print(">>> FAST <<<",flush=True)
# A fast run writes to a separate tag so it cannot overwrite the
# committed paper results.
_default_tag = "_partialfast" if os.environ.get("KVL_FAST","0")=="1" else "_partial"
TAG=os.environ.get("KVL_TAG", _default_tag)
print("CFG",CFG,"TAG",TAG,flush=True)

## 1. Load frozen base; probe GQA

In [ ]:
tok=AutoTokenizer.from_pretrained(BASE,cache_dir=CACHE)
if tok.pad_token is None: tok.pad_token=tok.eos_token
base=AutoModelForCausalLM.from_pretrained(BASE,cache_dir=CACHE,torch_dtype=torch.float32).to(DEVICE).eval()
for p in base.parameters(): p.requires_grad_(False)
ids=tok("hi",return_tensors="pt").input_ids.to(DEVICE)
with torch.no_grad(): pkv=base(ids,use_cache=True).past_key_values
N_LAYER=len(pkv); _,N_KV,_,D_HEAD=pkv[0][0].shape; KV_DIM=2*N_KV*D_HEAD
print(f"L={N_LAYER} n_kv={N_KV} d_head={D_HEAD} kv_dim={KV_DIM}",flush=True)

## 2. Cache tooling + reserved-last-token decode (verbatim from similar.ipynb)

In [ ]:
def kv_stack(past):
    outs=[]
    for (k,v) in past:
        B,H,T,dh=k.shape
        outs.append(torch.cat([k.permute(0,2,1,3).reshape(B,T,H*dh),
                               v.permute(0,2,1,3).reshape(B,T,H*dh)],-1))
    return torch.stack(outs,0)                       # (L,B,T,kv_dim)
def native_ids(text,max_len):
    return tok([text],return_tensors="pt",truncation=True,max_length=max_len).input_ids.to(DEVICE)
@torch.no_grad()
def ctx_cache(text,max_len):
    ids=native_ids(text,max_len); last=ids[:,-1:]; body=ids[:,:-1]
    leg=base(body,use_cache=True).past_key_values
    if not isinstance(leg,tuple): leg=leg.to_legacy_cache() if hasattr(leg,"to_legacy_cache") else leg
    return kv_stack(leg), last
def slots_to_legacy(vec):
    L,B,Kk,_=vec.shape; half=N_KV*D_HEAD; leg=[]
    for l in range(L):
        kt=vec[l,:,:,:half].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        vt=vec[l,:,:,half:].reshape(B,Kk,N_KV,D_HEAD).permute(0,2,1,3).contiguous()
        leg.append((kt,vt))
    return leg
def to_past(vec): return DynamicCache.from_legacy_cache(tuple(slots_to_legacy(vec)))
def _pos(s,l): return torch.arange(s,s+l,device=DEVICE).unsqueeze(0)
@torch.no_grad()
def greedy_from_prefix(prefix_vec,last_id):
    past=to_past(prefix_vec); T0=prefix_vec.shape[2]
    o=base(last_id,past_key_values=past,position_ids=_pos(T0,1),
           attention_mask=torch.ones(1,T0+1,device=DEVICE),use_cache=True)
    past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen=[cur]; L=T0+1
    for _ in range(CFG["gen_len"]-1):
        o=base(cur,past_key_values=past,position_ids=_pos(L,1),
               attention_mask=torch.ones(1,L+1,device=DEVICE),use_cache=True)
        past=o.past_key_values; cur=o.logits[:,-1,:].argmax(-1,keepdim=True); gen.append(cur); L+=1
    return tok.decode(torch.cat(gen,1)[0])

## 3. Split-context SQuAD (holder full / reader masked) — same probe

In [ ]:
def make_prompt(p,q): return f"{p}\nQuestion: {q}\nAnswer:"
def mask_answer(p,a):
    i=p.find(a); return None if i<0 else p[:i]+"____"+p[i+len(a):]
def build_qa(n,split):
    ds=datasets.load_dataset("squad",split=split,cache_dir=CACHE); items=[]
    for ex in ds:
        a=ex["answers"]["text"][0].strip()
        if not a or a not in ex["context"] or len(a.split())>4: continue
        m=mask_answer(ex["context"],a)
        if m is None: continue
        items.append(dict(q=ex["question"].strip(),full=ex["context"].strip(),masked=m.strip(),ans=a))
        if len(items)>=n: break
    return items
eval_qa=build_qa(CFG["n_eval"],"validation")
print("eval",len(eval_qa),flush=True)
def norm(s): return re.sub(r"[^a-z0-9 ]","",s.lower()).strip()
def hit(t,g): return norm(g) in norm(t) if g else False
@torch.no_grad()
def reader_self_kv(it): return ctx_cache(make_prompt(it["masked"],it["q"]),CFG["max_ctx"])
@torch.no_grad()
def reader_full_kv(it): return ctx_cache(make_prompt(it["full"],it["q"]),CFG["max_ctx"])
def cos_full(a,b): return F.cosine_similarity(a.reshape(-1),b.reshape(-1),dim=0).item()

## 4. Damage operators

`destroy(vec, mask)` replaces the masked coordinates with noise at **matched per-coordinate power**
(mean/std taken over the coordinates being destroyed), so a destroyed coordinate carries no information
about its true value but the tensor keeps its overall scale. Unmasked coordinates are left **bit-exact**.

This is deliberately the most favourable construction for the objection: the surviving features are not
merely close, they are *exactly* the reader's own.

In [ ]:
@torch.no_grad()
def destroy(vec, mask):
    # mask: broadcastable bool, True = destroy. Matched-power noise on destroyed coords.
    out=vec.clone()
    sel=mask.expand_as(vec)
    if sel.sum()==0: return out
    vals=vec[sel]
    mu=vals.mean(); sd=vals.std().clamp_min(1e-8)
    out[sel]=torch.randn_like(vals)*sd+mu
    return out

@torch.no_grad()
def mask_feature_frac(vec, p, gen):
    # random subset of the kv_dim FEATURE axis (same features across all layers/positions)
    L,B,T,Dk=vec.shape
    nsel=int(round(p*Dk))
    m=torch.zeros(Dk,dtype=torch.bool,device=vec.device)
    if nsel>0:
        idx=torch.randperm(Dk,generator=gen,device=vec.device)[:nsel]; m[idx]=True
    return m.view(1,1,1,Dk)

@torch.no_grad()
def mask_head(vec, head, which="k"):
    # one KV head (of N_KV) in the K half or V half
    L,B,T,Dk=vec.shape; half=N_KV*D_HEAD
    m=torch.zeros(Dk,dtype=torch.bool,device=vec.device)
    s=head*D_HEAD
    if which in ("k","both"): m[s:s+D_HEAD]=True
    if which in ("v","both"): m[half+s:half+s+D_HEAD]=True
    return m.view(1,1,1,Dk)

@torch.no_grad()
def mask_half(vec, which):
    L,B,T,Dk=vec.shape; half=N_KV*D_HEAD
    m=torch.zeros(Dk,dtype=torch.bool,device=vec.device)
    if which=="k": m[:half]=True
    else:          m[half:]=True
    return m.view(1,1,1,Dk)

@torch.no_grad()
def destroy_layers(vec, layers):
    out=vec.clone()
    for l in layers:
        sub=out[l:l+1]
        mu=sub.mean(); sd=sub.std().clamp_min(1e-8)
        out[l:l+1]=torch.randn_like(sub)*sd+mu
    return out

@torch.no_grad()
def noisy_to_cos(vec, target):
    # dense Arm-A construction, for the matched-cosine control
    n=torch.randn_like(vec)
    sig_pow=(vec**2).mean().sqrt(); n=n/(n**2).mean().sqrt()*sig_pow
    r=float((1.0/target**2 - 1.0)**0.5)
    return vec + r*n

## 5. LEG P1 + P2 — feature-subset damage, with achieved cosine and a matched dense control

For each `p`: destroy that fraction of features (rest exact), record accuracy and the achieved global
cosine. Then build a **dense** perturbation calibrated to that *same* cosine and record its accuracy.
Averaged over seeds.

In [ ]:
@torch.no_grad()
def leg_P1P2(data):
    fr=CFG["fracs"]
    acc={f"p{p}":0.0 for p in fr}; cosa={f"p{p}":[] for p in fr}
    acc_dense={f"p{p}":0.0 for p in fr}
    floor=0.0; ceiling=0.0; n=0
    for it in data:
        g=it["ans"]
        own,last=reader_self_kv(it)
        full,lastf=reader_full_kv(it)
        floor  += hit(greedy_from_prefix(own,last),g)
        ceiling+= hit(greedy_from_prefix(full,lastf),g)
        for p in fr:
            a_s=[]; c_s=[]; ad_s=[]
            for s in CFG["seeds"]:
                gen=torch.Generator(device=own.device); gen.manual_seed(1000*s+7)
                m=mask_feature_frac(own,p,gen)
                dmg=destroy(own,m)
                c=cos_full(own,dmg); c_s.append(c)
                a_s.append(hit(greedy_from_prefix(dmg,last),g))
                # matched-cosine DENSE control
                ct=min(max(c,1e-4),0.999999)
                dense=noisy_to_cos(own,ct)
                ad_s.append(hit(greedy_from_prefix(dense,last),g))
            acc[f"p{p}"]       += float(np.mean(a_s))
            acc_dense[f"p{p}"] += float(np.mean(ad_s))
            cosa[f"p{p}"].append(float(np.mean(c_s)))
        n+=1
    out=dict(acc={k:v/max(1,n) for k,v in acc.items()},
             acc_dense_matched={k:v/max(1,n) for k,v in acc_dense.items()},
             achieved_cos={k:float(np.mean(v)) for k,v in cosa.items()},
             floor=floor/max(1,n), ceiling=ceiling/max(1,n))
    return out
P1=leg_P1P2(eval_qa)
print("\n=== LEG P1/P2: FEATURE-SUBSET DAMAGE (rest bit-exact) ===",flush=True)
print(f"  floor={P1['floor']:.3f}  ceiling={P1['ceiling']:.3f}",flush=True)
print(f"  {'frac damaged':>13} {'ach.cos':>9} {'sparse acc':>11} {'dense@same cos':>15}",flush=True)
for p in CFG["fracs"]:
    print(f"  {p:>13} {P1['achieved_cos'][f'p{p}']:>9.4f} {P1['acc'][f'p{p}']:>11.3f} "
          f"{P1['acc_dense_matched'][f'p{p}']:>15.3f}",flush=True)

## 6. LEG P3 — structural subsets: exactly one KV head, exactly one layer

Same total-destruction operator, but the damaged unit is a whole head or a whole layer. `one layer` is the
direct tie-in to the depth experiment: if destroying a single layer out of 28 already lands at/below the
floor, the wall is not about how much of the cache is wrong.

In [ ]:
@torch.no_grad()
def leg_P3(data):
    res={}
    heads=list(range(N_KV))
    layer_probe=[0,1,N_LAYER//2,N_LAYER-1]
    acc={f"head{h}_k":0.0 for h in heads}
    acc.update({f"head{h}_v":0.0 for h in heads})
    acc.update({f"layer{l}":0.0 for l in layer_probe})
    cosr={k:[] for k in acc}
    n=0
    for it in data:
        g=it["ans"]; own,last=reader_self_kv(it)
        for h in heads:
            for w in ("k","v"):
                m=mask_head(own,h,w); d=destroy(own,m)
                acc[f"head{h}_{w}"]+=hit(greedy_from_prefix(d,last),g)
                cosr[f"head{h}_{w}"].append(cos_full(own,d))
        for l in layer_probe:
            d=destroy_layers(own,[l])
            acc[f"layer{l}"]+=hit(greedy_from_prefix(d,last),g)
            cosr[f"layer{l}"].append(cos_full(own,d))
        n+=1
    return dict(acc={k:v/max(1,n) for k,v in acc.items()},
                cos={k:float(np.mean(v)) for k,v in cosr.items()},
                layer_probe=layer_probe, n_kv=N_KV, n_layer=N_LAYER)
P3=leg_P3(eval_qa)
print("\n=== LEG P3: STRUCTURAL SUBSETS (one head / one layer destroyed) ===",flush=True)
for k in sorted(P3["acc"]):
    print(f"  {k:>10}  cos={P3['cos'][k]:.4f}  acc={P3['acc'][k]:.3f}",flush=True)

## 7. LEG P4 — K half vs V half at matched fraction

Destroy a fraction `p` of features drawn only from the K half, versus only from the V half. The
softmax-consumption-point account predicts K damage is materially worse: keys enter the normalized
`q.k` competition, values are averaged under weights that are already fixed.

In [ ]:
@torch.no_grad()
def leg_P4(data):
    fr=[0.01,0.05,0.10,0.25]
    acc={f"{w}_p{p}":0.0 for w in ("k","v") for p in fr}
    cosr={k:[] for k in acc}
    n=0; half=N_KV*D_HEAD
    for it in data:
        g=it["ans"]; own,last=reader_self_kv(it)
        Dk=own.shape[-1]
        for w in ("k","v"):
            lo,hi=(0,half) if w=="k" else (half,Dk)
            for p in fr:
                a_s=[]; c_s=[]
                for s in CFG["seeds"]:
                    gen=torch.Generator(device=own.device); gen.manual_seed(2000*s+11)
                    span=hi-lo; nsel=int(round(p*span))
                    m=torch.zeros(Dk,dtype=torch.bool,device=own.device)
                    if nsel>0:
                        idx=torch.randperm(span,generator=gen,device=own.device)[:nsel]+lo
                        m[idx]=True
                    d=destroy(own,m.view(1,1,1,Dk))
                    a_s.append(hit(greedy_from_prefix(d,last),g)); c_s.append(cos_full(own,d))
                acc[f"{w}_p{p}"]+=float(np.mean(a_s)); cosr[f"{w}_p{p}"].append(float(np.mean(c_s)))
        n+=1
    return dict(acc={k:v/max(1,n) for k,v in acc.items()},
                cos={k:float(np.mean(v)) for k,v in cosr.items()}, fracs=fr)
P4=leg_P4(eval_qa)
print("\n=== LEG P4: K-half vs V-half damage ===",flush=True)
for p in P4["fracs"]:
    print(f"  p={p:<5} K: cos={P4['cos'][f'k_p{p}']:.4f} acc={P4['acc'][f'k_p{p}']:.3f}   "
          f"V: cos={P4['cos'][f'v_p{p}']:.4f} acc={P4['acc'][f'v_p{p}']:.3f}",flush=True)

## 8. Save + plot

In [ ]:
RES=dict(config=CFG,base=BASE,
         P1_feature_frac=P1,P3_structural=P3,P4_kv_split=P4,
         anchors=dict(floor=P1["floor"],ceiling=P1["ceiling"]))
out=f"results/partial{TAG}.json"
json.dump(RES,open(out,"w"),indent=1); print("saved",out,flush=True)

fr=CFG["fracs"]
fig,ax=plt.subplots(1,3,figsize=(16,4.2))
xs=[p for p in fr]
ax[0].plot(xs,[P1["acc"][f"p{p}"] for p in fr],"o-",color="#C0392B",label="sparse (rest bit-exact)")
ax[0].plot(xs,[P1["acc_dense_matched"][f"p{p}"] for p in fr],"s--",color="#2E75B6",
           label="dense @ same cosine")
ax[0].axhline(P1["floor"],ls="--",c="#7f8c8d",label=f"floor {P1['floor']:.2f}")
ax[0].axhline(P1["ceiling"],ls=":",c="#27AE60",label=f"ceiling {P1['ceiling']:.2f}")
ax[0].set_xscale("symlog",linthresh=0.01)
ax[0].set_xlabel("fraction of features destroyed"); ax[0].set_ylabel("fact-transfer accuracy")
ax[0].set_title("P1/P2: does keeping most features exact help?"); ax[0].legend(fontsize=8)

ks=[f"layer{l}" for l in P3["layer_probe"]]+[f"head{h}_k" for h in range(P3["n_kv"])]+\
   [f"head{h}_v" for h in range(P3["n_kv"])]
ax[1].bar(range(len(ks)),[P3["acc"][k] for k in ks],color="#C0392B")
ax[1].axhline(P1["floor"],ls="--",c="#7f8c8d"); ax[1].axhline(P1["ceiling"],ls=":",c="#27AE60")
ax[1].set_xticks(range(len(ks))); ax[1].set_xticklabels(ks,rotation=45,ha="right",fontsize=7)
ax[1].set_ylabel("accuracy"); ax[1].set_title("P3: one layer / one head destroyed")

for w,c in (("k","#C0392B"),("v","#2E75B6")):
    ax[2].plot(P4["fracs"],[P4["acc"][f"{w}_p{p}"] for p in P4["fracs"]],"o-",color=c,label=f"{w.upper()} half")
ax[2].axhline(P1["floor"],ls="--",c="#7f8c8d"); ax[2].set_xlabel("fraction destroyed within half")
ax[2].set_ylabel("accuracy"); ax[2].set_title("P4: K vs V"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("figures/partial_fig.png",dpi=140)
plt.show()

## 9. Verdict

Read the three panels as one answer to the objection:

* **If P1 sparse tracks `(1-p)` gracefully** the reviewer is right and the headline result is an artifact
  of dense perturbation. The paper's claim would need to be narrowed.
* **If P1 sparse craters at the smallest `p`** — while `1-p` of the cache is bit-exact — then "most
  features were good" is no defence, and the near-zero-radius claim is strengthened rather than
  weakened. P2 then also shows sparse-severe is worse than dense-mild *at equal cosine*, which is a
  second, independent demonstration that cosine is the wrong instrument.
* **P3** says whether the unit of failure is structural (a head, a layer) rather than a coordinate count.
* **P4** tests the mechanism directly: K worse than V is what the softmax account predicts.